<a href="https://colab.research.google.com/github/hsultova/Softuni-AI-Agents-Workflows/blob/main/langChain_agent_tools.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q langchain langchain-openai langchain-text-splitters chromadb  langchain_chroma

In [139]:
import json
from typing import List, Callable

from google.colab import userdata
from langchain.agents import create_agent, AgentState
from langchain_openai import ChatOpenAI
from langchain_core.documents import Document
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_chroma import Chroma
from langchain.tools import tool
from langchain_core.tools import create_retriever_tool
from langchain_core.messages import ToolMessage
from langchain.messages import HumanMessage, ToolMessage
from langchain.agents.middleware import PIIMiddleware, ModelRequest, ModelResponse, ToolCallRequest, after_agent, after_model, before_agent, before_model, wrap_model_call, wrap_tool_call
from langgraph.runtime import Runtime

# Agent and Model




In [92]:
openai_api_key = userdata.get('OPENAI_API_KEY')
openai_model = ChatOpenAI(model="gpt-4.1-mini", api_key=openai_api_key)

In [79]:
agent = create_agent(model=openai_model)

#  Building the Knowledge Base from Local Text

# Loader

In [4]:
FAQ_PATH = '/content/FAQ.md'
with open(FAQ_PATH) as file:
    text = file.read()
documents = [Document(page_content=text, metadata={"source": FAQ_PATH})]

In [5]:
documents

[Document(metadata={'source': '/content/FAQ.md'}, page_content='# Store Technical Documentation — Frequently Asked Questions (FAQ)\n\n## Q: What is the warranty period?\nA: 24 months for individuals (consumers) and 12 months for legal entities (businesses), in accordance with applicable consumer protection legislation.\n\n## Q: What does the warranty cover?\nA: The warranty covers manufacturing defects, material faults, and malfunctions that occur under normal use of the product. It does not cover damage caused by misuse, accidents, unauthorized repairs, or normal wear and tear.\n\n## Q: What is not covered by the warranty?\nA: Mechanical damage caused by the customer, liquid damage, damage from improper storage or use, cosmetic wear (scratches, dents), consumables (batteries, cables if worn out), and issues caused by third-party repairs or modifications.\n\n## Q: How do I claim a warranty repair or replacement?\nA: Contact our customer support with your order number and a description 

# Splitter

In [6]:
headers_to_split_on = [
    ("##", "Question")
]
splitter = MarkdownHeaderTextSplitter(headers_to_split_on, strip_headers=False)

In [7]:
for document in documents:
  print(splitter.split_text(document.page_content))

[Document(metadata={'Question': 'Q: What is the warranty period?'}, page_content='# Store Technical Documentation — Frequently Asked Questions (FAQ)  \n## Q: What is the warranty period?\nA: 24 months for individuals (consumers) and 12 months for legal entities (businesses), in accordance with applicable consumer protection legislation.'), Document(metadata={'Question': 'Q: What does the warranty cover?'}, page_content='## Q: What does the warranty cover?\nA: The warranty covers manufacturing defects, material faults, and malfunctions that occur under normal use of the product. It does not cover damage caused by misuse, accidents, unauthorized repairs, or normal wear and tear.'), Document(metadata={'Question': 'Q: What is not covered by the warranty?'}, page_content='## Q: What is not covered by the warranty?\nA: Mechanical damage caused by the customer, liquid damage, damage from improper storage or use, cosmetic wear (scratches, dents), consumables (batteries, cables if worn out), an

In [8]:
chunks = []
for document in documents:
  chunks.extend(splitter.split_text(document.page_content))

# Vector store: Chroma DB

In [12]:
chroma = Chroma(collection_name="faq_db", persist_directory="/content/chroma")

In [13]:
chroma.add_documents(chunks)

['3d6526d5-4b19-4599-9101-abc3c9a3c1e2',
 '9cb9b240-808c-4038-b759-face8f355c52',
 '1d55b6ef-55ce-408f-9c01-508cb2231173',
 'bd81cf45-559d-440c-9f15-6fba3d5613ed',
 '5263960b-a671-4278-b428-ec4ac9d0dbdf',
 '0446f860-3391-43f2-b326-bf51a3cd953d',
 '4e7862f6-b65c-4626-8290-1069fb8e0d00',
 'fee464f2-b2d1-4b48-b727-5d17f05446cb',
 'db4acd6c-42d6-446b-aa4e-d0ca7bd0732e',
 '85498b85-562e-421f-9791-4c8d738d2d66',
 'c1af7b11-3e23-44ff-be3c-ffb4c2e66c79',
 'fce28c7a-0fc0-4a9b-aac9-dc7cd098499c',
 'e40caca2-47dd-46de-bc51-34d514122329',
 '491427dc-c031-449b-9448-5095644f9cc5',
 '2dee46ed-db9a-4218-ba86-07c200816f0b',
 'f370c0ae-ded8-4b42-bb09-cd7a7ae771f5',
 '56a6bbae-4af4-4002-86a2-78f5d896b590',
 'acad85dd-63ab-4a53-b609-a79bf5add4de',
 'bd722668-6d03-4705-956f-ec52a57fe9ce',
 'aa22554d-e0ff-4a6f-b6cf-91faff123d0f',
 '36ae15d9-a25a-4b1d-b0c3-4ec3cbf5657d',
 '627fbd76-82f7-44db-95e9-20f3f4cced3f',
 'cf8892f9-419d-4a6a-b178-1050d574a06c',
 '3148f338-1fa0-4866-ac5a-e138eadb8715',
 '7ad19cd8-2795-

# Retriever
Transorm the db to retriever-tool

In [15]:
faq_retriever = chroma.as_retriever()

In [16]:
faq_retriever.invoke("warranty")

[Document(id='3d6526d5-4b19-4599-9101-abc3c9a3c1e2', metadata={'Question': 'Q: What is the warranty period?'}, page_content='# Store Technical Documentation — Frequently Asked Questions (FAQ)  \n## Q: What is the warranty period?\nA: 24 months for individuals (consumers) and 12 months for legal entities (businesses), in accordance with applicable consumer protection legislation.'),
 Document(id='1d55b6ef-55ce-408f-9c01-508cb2231173', metadata={'Question': 'Q: What is not covered by the warranty?'}, page_content='## Q: What is not covered by the warranty?\nA: Mechanical damage caused by the customer, liquid damage, damage from improper storage or use, cosmetic wear (scratches, dents), consumables (batteries, cables if worn out), and issues caused by third-party repairs or modifications.'),
 Document(id='bd81cf45-559d-440c-9f15-6fba3d5613ed', metadata={'Question': 'Q: How do I claim a warranty repair or replacement?'}, page_content='## Q: How do I claim a warranty repair or replacement?\

# Tools

# Connecting to Internal Systems

In [25]:
orders = [
  { "id": 1, "status": "pending" },
  { "id": 2, "status": "shipped" },
  { "id": 3, "status": "delivered" },
  { "id": 4, "status": "cancelled" },
  { "id": 5, "status": "processing" }
]

In [74]:
@tool
def customer_orders() -> str:
  """
  Call this tool to retrieve all orders of the customer.
  """
  return json.dumps(orders)

@tool
def lookup_order(id: int) -> str:
  """
  Call this tool to lookup additional information about an order by given id.
  """
  order = next((item for item in orders if item["id"] == int(id)), None)
  return order

In [96]:
faq_retriever_tool = create_retriever_tool(faq_retriever, name="lookup_knowledge", description="Use this tool to look up text to find the exact sentence needed to answer a customer's question." )

# Middleware

In [147]:
@before_agent
def before_agent_func(state: AgentState, runtime: Runtime) -> None:
    print("Event: before_agent")
    print(state)
    print(runtime)

@before_model
def before_model_func(state: AgentState, runtime: Runtime) -> None:
    print("Event: before_model")
    print(state)
    print(runtime)

@after_model
def after_model_func(state: AgentState, runtime: Runtime) -> None:
    print("Event: after_model")
    print(state)
    print(runtime)

@after_agent
def after_agent_func(state: AgentState, runtime: Runtime) -> None:
    print("Event: after_agent")
    print(state)
    print(runtime)

### @wrap_model_call
# def handle_model_call(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]):
#     print("Event: model_call")
#     print(request)

#     # # Only force tool on the first LLM step
#     # if len(request.messages) == 1:
#     #     print("Forcing a specific tool call")
#     #     request = request.override(tool_choice=faq_retriever_tool.name)

#     response = handler(request)

#     print("Obtained model response:")
#     print(response)

#     return response

### @wrap_tool_call
# def handle_tool_call(request: ToolCallRequest, handler: Callable[[ToolCallRequest], ToolMessage]):
#     print("Event: tool_call")
#     print(request)

#     response = handler(request)

#     print("Obtained tool message:")
#     print(response)

# Setting up the "Brain" (The Agent)

In [137]:
def print_conversation(conversation: List[BaseMessage]):
    for message in conversation:
        message.pretty_print()

In [148]:
customer_support_agent = create_agent(
    model=openai_model,
    tools = [faq_retriever_tool, customer_orders, lookup_order],
    system_prompt="You are a helpful customer support agent",
    middleware=[
        PIIMiddleware(pii_type="email", strategy="redact"),
        before_agent_func,
        before_model_func,
        after_agent_func,
        after_model_func
    ])

In [149]:
delivery_response = customer_support_agent.invoke(input={"messages": [HumanMessage("How long I should wait for the standard delievery?")]})

Event: before_agent
{'messages': [HumanMessage(content='How long I should wait for the standard delievery?', additional_kwargs={}, response_metadata={}, id='e1cd775e-182f-4b1a-bc42-aa486f52d53b')]}
Runtime(context=None, store=None, stream_writer=<function Pregel.stream.<locals>.stream_writer at 0x7f5248a9a200>, heartbeat=<function _no_op_heartbeat at 0x7f52cd7d54e0>, previous=None, execution_info=ExecutionInfo(checkpoint_id='1f1a6015-f133-65b3-8000-30d8d9b1aeb5', checkpoint_ns='before_agent_func.before_agent:6e897762-b72e-901d-5b86-9d25a1ecff28', task_id='6e897762-b72e-901d-5b86-9d25a1ecff28', thread_id=None, run_id=None, node_attempt=1, node_first_attempt_time=1788266024.5473437), server_info=None, control=<langgraph.runtime.RunControl object at 0x7f5248a6e680>)
Event: before_model
{'messages': [HumanMessage(content='How long I should wait for the standard delievery?', additional_kwargs={}, response_metadata={}, id='e1cd775e-182f-4b1a-bc42-aa486f52d53b')]}
Runtime(context=None, store=

In [117]:
print_conversation(delivery_response['messages'])

================================ Human Message =================================

How long I should wait for the standard delievery?
================================== Ai Message ==================================
Tool Calls:
  lookup_knowledge (call_8YbRniNU2XvxCjfntUuJWCdU)
 Call ID: call_8YbRniNU2XvxCjfntUuJWCdU
  Args:
    query: standard delivery time
================================= Tool Message =================================
Name: lookup_knowledge

## Q: How long does delivery take?
A: Standard delivery: [X-X] business days. Express delivery: [X] business day(s). Delivery times may vary for remote areas.

## Q: What shipping methods are available?
A: Courier delivery, pickup from a physical store location, and delivery to partner pickup points/lockers.

## Q: Do you ship internationally?
A: [Yes/No] — specify countries served, customs responsibilities, and any additional fees or delays.

## Q: Can I modify or cancel an order after placing it?
A: Orders can be modified or can

In [108]:
order_response = customer_support_agent.invoke(input={"messages": [HumanMessage("What is the status of the order with id 1")]})

In [109]:
print_conversation(order_response['messages'])

================================ Human Message =================================

What is the status of the order with id 1
================================== Ai Message ==================================
Tool Calls:
  lookup_order (call_l1MS3IIC4Bjg9EUC2qWuwlAQ)
 Call ID: call_l1MS3IIC4Bjg9EUC2qWuwlAQ
  Args:
    id: 1
================================= Tool Message =================================
Name: lookup_order

{"id": 1, "status": "pending"}
================================== Ai Message ==================================

The status of the order with ID 1 is pending. Is there anything else you would like to know about this order?


In [150]:
return_order_response = customer_support_agent.invoke(input={"messages": [HumanMessage("Check status of the order 2 and tell me if I can return it?")]})

Event: before_agent
{'messages': [HumanMessage(content='Check status of the order 2 and tell me if I can return it?', additional_kwargs={}, response_metadata={}, id='5e8358c8-b480-48a8-b31a-359db147f940')]}
Runtime(context=None, store=None, stream_writer=<function Pregel.stream.<locals>.stream_writer at 0x7f5248a3c400>, heartbeat=<function _no_op_heartbeat at 0x7f52cd7d54e0>, previous=None, execution_info=ExecutionInfo(checkpoint_id='1f1a6018-8a9a-69c0-8000-76e0de5efd82', checkpoint_ns='before_agent_func.before_agent:6b102b0b-f0da-1b24-165f-a4dbf28298e0', task_id='6b102b0b-f0da-1b24-165f-a4dbf28298e0', thread_id=None, run_id=None, node_attempt=1, node_first_attempt_time=1788266094.3196673), server_info=None, control=<langgraph.runtime.RunControl object at 0x7f5248a9dcc0>)
Event: before_model
{'messages': [HumanMessage(content='Check status of the order 2 and tell me if I can return it?', additional_kwargs={}, response_metadata={}, id='5e8358c8-b480-48a8-b31a-359db147f940')]}
Runtime(co

In [151]:
print_conversation(return_order_response['messages'])

================================ Human Message =================================

Check status of the order 2 and tell me if I can return it?
================================== Ai Message ==================================
Tool Calls:
  lookup_order (call_LUVuwxAGDJmeBHDE5DYmappC)
 Call ID: call_LUVuwxAGDJmeBHDE5DYmappC
  Args:
    id: 2
================================= Tool Message =================================
Name: lookup_order

{"id": 2, "status": "shipped"}
================================== Ai Message ==================================
Tool Calls:
  lookup_knowledge (call_GA0LCoa9uqXDKOOPzL267nRC)
 Call ID: call_GA0LCoa9uqXDKOOPzL267nRC
  Args:
    query: return policy for order
================================= Tool Message =================================
Name: lookup_knowledge

## Q: How do I initiate a return?
A: Submit a return request through your account or contact customer support with your order number. You will receive instructions and a return shipping label (if 